## **Setup & Load Data**

In [ ]:
# install.packages("FNN")
library(FNN)
library(lubridate)
library(tidyverse)
library(readr)
library(stringr)
library(bigrquery)
library(parallel)

In [ ]:
EXPORT_BUCKET = "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055"
BILLING = "wb-silky-pepper-6055"
DATA_MOUNT = "/home/jupyter/workspace/raw/vwb-aou-datasets-controlled/v8"
# CDR_STORAGE_PATH = "gs://fc-aou-datasets-controlled/v8"
# WORKSPACE_CDR = "wb-silky-artichoke-2408.C2024Q3R8"
# previous proj = "terra-vpc-sc-d3cc1fbe"

In [ ]:
# Read query data directly from Cloud Storage into memory
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

# Read a single file from cloud storage into memory
# gs_cat <- function(gs_path) {
#   if (nzchar(BILLING)) pipe(sprintf("gsutil -u %s cat '%s'", BILLING, gs_path))
#   else pipe(sprintf("gsutil cat '%s'", gs_path))
# }


In [ ]:
person_path <- "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055/bq_exports/person/20260426/person/person_*.csv"
person_df <- read_bq_export_from_workspace_bucket(person_path)

measurementOccurrence_path <- "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055/bq_exports/measurementOccurrence/20260426/measurementOccurrence/measurementOccurrence_*.csv"
measure_df <- read_bq_export_from_workspace_bucket(measurementOccurrence_path)

In [ ]:
# Read ancestry predictions
#TODO: why does this workspace have echo_v4_r2 prefix?
ancestry_path = paste0(DATA_MOUNT, "/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv")
ancestry_raw <- read_tsv(
  ancestry_path,
  show_col_types = FALSE,
  progress = FALSE)

## **Get Analytical Dataset**

For now, will only include:

HbA1c numeric
HbA1c >= 3 & <= 12
Participants with 3 or more HbA1c measures
Participants with ancestry PCs
TODO: not sure if correct to summarize with multiple values for individuals present...

TODO: re-add summary plots elsewhere

In [ ]:
# 1) Pull and clean HbA1c values
#TODO: add formal NA removal here
#TODO: re-add confirmation all UTC here?
head(measure_df)
hba1c <- measure_df %>%
  transmute(
      hba1c = as.numeric(value_as_number),
      person_id = as.character(format(person_id, scientific = FALSE, trim = TRUE)),
      datetime = ymd_hms(measurement_datetime, tz = "UTC")) %>%
  filter(is.finite(hba1c)) %>%
  filter(hba1c >= 3, hba1c <= 12)

# Find IDs with >= 3 readings
ids_3plus <- hba1c %>%
  count(person_id) %>%
  filter(n >= 3) %>%
  pull(person_id)

In [ ]:
head(hba1c)

In [ ]:
# ----------------------------
# 2) Load ancestry PCs and keep only cohort participants
# ----------------------------
pc_cols <- paste0("pc", 1:16)
ancestry_pc <- ancestry_raw %>%
  transmute(
    research_id = as.character(format(research_id, scientific = FALSE, trim = TRUE)),
    # ancestry_pred_other = if ("ancestry_pred_other" %in% names(ancestry_raw)) ancestry_pred_other else NA_character_,
    pca_str = str_remove_all(pca_features, "\\[|\\]")
  ) %>%
  separate(
    pca_str,
    into = pc_cols,
    sep = ",\\s*",
    convert = TRUE,
    remove = TRUE
  )

In [ ]:
# Filter to match hba1c cohort above
ancestry_in_cohort <- ancestry_pc %>%
    dplyr::filter(research_id %in% ids_3plus)

# Filter to ensure above have PCs and 3+ measurements
hba1c_anal <- hba1c %>%
    dplyr::filter(person_id %in% ancestry_in_cohort$research_id) %>%
    dplyr::filter(person_id %in% ids_3plus)

ids_anal <- unique(hba1c_anal$person_id)

# Summary of analytical dataset
cat("Number valid HbA1C measurements:", nrow(hba1c), "\n")
cat("Participants with valid HbA1C:", n_distinct(hba1c$person_id), "\n")
cat("Participants with >= 3 readings:", length(ids_3plus), "\n")
cat("Number valid HbA1C measurements after subset to >= 3 readings:", nrow(hba1c_anal), "\n")
cat("Participants with valid HbA1C, >= 3 readings, & srWGS ancestry PCs:", nrow(ancestry_in_cohort), "\n")

## **Get Matched Cohorts**

Alternate method for GenoSiS = using Euclidian distance on 16 ancestry PCs.

In [ ]:
# # Create the numeric matrix for fast distance calculation
# # Rows = people, Cols = PC1..PC16
# pc_matrix <- as.matrix(ancestry_in_cohort %>% column_to_rownames("research_id"))

In [ ]:
# k_neighbors <- 100
# # restrict_same_ancestry <- FALSE  # set TRUE to only search within same ancestry_pred_other
# #TODO: do I want to force matches to be within same ancestry group? - currently no

# knn_raw <- get.knn(pc_matrix, k=k_neighbors, algorithm="kd_tree")
# knn_cohort <- matrix(
#   rownames(pc_matrix)[knn_raw$nn.index],  # get id at index from input rownames
#   nrow = nrow(knn_raw$nn.index))  # same number rows as raw output
# rownames(knn_cohort) <- rownames(pc_matrix)  # add id that neighbors were found for

# # #TODO: do I want to save the actual distances too? - currently no

In [ ]:
# head(knn_cohort)
# dim(knn_cohort)

In [ ]:
# knn_cohort_t <- t(knn_cohort)
# head(knn_cohort_t)
# dim(knn_cohort_t)

In [ ]:
# Save file
# write_tsv(as.data.frame(knn_cohort_t), "matched_cohorts.txt")
# system(paste0("gsutil cp matched_cohorts.txt ", EXPORT_BUCKET, "/"))

In [ ]:
# Reload file
system(paste0("gsutil cp ", EXPORT_BUCKET, "/matched_cohorts.txt ./"))
knn_cohort_t <- read_tsv("matched_cohorts.txt")
head(knn_cohort_t)
dim(knn_cohort_t)

## **Analysis**

In [ ]:
# Definitions re-used throughout
prediab <- 5.7
diab <- 6.5

#### **Mean Shift for Every Individual**

Mean shift from pre-diabetes and diabetes cutoffs is calculated by using the shift in maxmimum density in the whole cohort compared to the matched cohort.

In [ ]:
# Global Density Peak
dim(hba1c_anal)
full_d <- density(hba1c_anal$hba1c, na.rm = TRUE)
full_d_peak <- full_d$x[which.max(full_d$y)]

summary(hba1c_anal$hba1c)
print(full_d_peak)

In [ ]:
# For every person
    # Calculate their cohort's maximum A1c density
    # Calculate the shifted thresholds
get_shift <- function(id, cohort_ids) {
    cohort_ids <- unlist(cohort_ids, use.names = F)
    
    cohort <- hba1c_anal[hba1c_anal$person_id %in% cohort_ids, ]
    #TODO: could add average number measures per person in each cohort

    cohort_d <- density(cohort$hba1c)  # checks on NA done & range already set
    cohort_d_peak <- cohort_d$x[which.max(cohort_d$y)]
    #TODO: read more about whether different density bandwidth should be used?

    shift <- full_d_peak - cohort_d_peak
    return(shift)
}

In [ ]:
#TO NOTE: these two rows must use the same ID list based on order
neighbor_list <- lapply(ids_anal, function(id) knn_cohort_t[, id])

In [ ]:
# # detectCores()  # 4
# # TO NOTE: this takes 20+ minutes as currently written

# shift_values <- mclapply(seq_along(ids_anal), function(i) {
#   id <- ids_anal[i]
#   cohort_ids <- neighbor_list[[i]]
#   get_shift(id, cohort_ids)
# }, mc.cores = 4)

# df <- data.frame(id = ids_anal, shift = unlist(shift_values))

In [ ]:
# # Save file
# write_tsv(df, "hba1c_anal_shift.txt")
# system(paste0("gsutil cp hba1c_anal_shift.txt ", EXPORT_BUCKET, "/"))

In [ ]:
# Reload file
system(paste0("gsutil cp ", EXPORT_BUCKET, "/hba1c_anal_shift.txt ./"))
df <- read.table("hba1c_anal_shift.txt", header = TRUE)
dim(df)
head(df)

In [ ]:
df_mod <- df %>%
    dplyr::mutate(
        id = as.character(format(id, scientific = FALSE, trim = TRUE))) %>%
    dplyr::mutate(
        pers_prediab = prediab - shift,
        pers_diab = diab - shift)

In [ ]:
summary(df_mod$shift)
summary(df_mod$pers_prediab)
summary(df_mod$pers_diab)

#### **Measurement-Level Confusion Matrix**

TO NOTE: this preliminary version is based only on measurement thresholds, does not include ICD codes.

In [ ]:
# Think will first do this for all measures for every individual
# 1. classify each measure based on standard thresholds
# 2. classify each measure based on personalized thresholds
hba1c_anal_thresh <- inner_join(
    hba1c_anal,
    df_mod,
    by = c("person_id" = "id")
)

In [ ]:
dim(hba1c_anal_thresh)
head(hba1c_anal_thresh)

In [ ]:
# Make confusion matrix
hba1c_anal_thresh <- hba1c_anal_thresh %>%
    dplyr::mutate(conf_prediab = case_when(
        hba1c < prediab & hba1c < pers_prediab ~ "tn",
        hba1c < prediab & hba1c >= pers_prediab ~ "fn",
        hba1c >= prediab & hba1c < pers_prediab ~ "fp",
        hba1c >= prediab & hba1c >= pers_prediab ~ "tp"
    )) %>%
    dplyr::mutate(conf_diab = case_when(
        hba1c < diab & hba1c < pers_diab ~ "tn",
        hba1c < diab & hba1c >= pers_diab ~ "fn",
        hba1c >= diab & hba1c < pers_diab ~ "fp",
        hba1c >= diab & hba1c >= pers_diab ~ "tp"
    ))

In [ ]:
table(hba1c_anal_thresh$conf_prediab)
table(hba1c_anal_thresh$conf_diab)

#### **Individuals with False Negatives**

Stuck on best way to convert to individual level...

For now, subset to individuals with at least one false negative measurement, and summarize whether under the standard thresholds they:

would never have been diagnosed ("fn" but never a "tp")
would have been diagnosed late ("fn" and "tp")

In [ ]:
prediab_fn_summ <- hba1c_anal_thresh %>%
  dplyr::group_by(person_id) %>%
  dplyr::summarize(
      count_fn = sum(conf_prediab == "fn"),
      count_tp = sum(conf_prediab == "tp"))

diab_fn_summ <- hba1c_anal_thresh %>%
  dplyr::group_by(person_id) %>%
  dplyr::summarize(
      count_fn = sum(conf_diab == "fn"),
      count_tp = sum(conf_diab == "tp"))

In [ ]:
nrow(prediab_fn_summ)
n_distinct(prediab_fn_summ$person_id)
sum(prediab_fn_summ$count_fn >= 1 & prediab_fn_summ$count_tp >= 1)
sum(prediab_fn_summ$count_fn >= 1 & prediab_fn_summ$count_tp == 0)

In [ ]:
sum(diab_fn_summ$count_fn >= 1 & diab_fn_summ$count_tp >= 1)
sum(diab_fn_summ$count_fn >= 1 & diab_fn_summ$count_tp == 0)

#### **Updated Individual-Level Summary**

Define true early warning: pers-diab before diab OR pers-diab and never diab

Need to get each person's earliest date of:

+ pre-diab
+ diab
+ pers-pre-diab
+ pers-diab
+ normal?

In [ ]:
# Modifying code from Hayley
date_summ <- hba1c_anal_thresh %>%
  mutate(
    pers = case_when(
      hba1c < pers_prediab ~ "normal",
      hba1c < pers_diab ~ "prediab",
      TRUE ~ "diab"
    ),
    classic = case_when(
      hba1c < prediab ~ "normal",
      hba1c < diab ~ "prediab",
      TRUE ~ "diab"
    )) %>%
  pivot_longer(
    cols = c(pers, classic),
    names_to = "type",
    values_to = "category"
  ) %>%
  filter(category %in% c("prediab", "diab")) %>%
  group_by(person_id, type, category) %>%
  summarise(first_date = min(datetime), .groups = "drop") %>%
  mutate(date_col = paste0(category, "_date_", type)) %>%
  select(person_id, date_col, first_date) %>%
  pivot_wider(
    names_from = date_col,
    values_from = first_date
  )

In [ ]:
# Summary
cat("Distinct individuals:", n_distinct(hba1c_anal_thresh$person_id), "\n")
cat("Distinct individuals with only normal classic/personalized:", length(setdiff(unique(hba1c_anal_thresh$person_id), unique(date_summ$person_id))), "\n")
cat("Distinct individuals with classic/personalized prediab/diab:", n_distinct(date_summ$person_id), "\n")

In [ ]:
early_warning_summ <- date_summ %>%
  mutate(
    prediab_true_early =
      !is.na(prediab_date_pers) &
      (
        is.na(prediab_date_classic) |
        prediab_date_pers < prediab_date_classic
      ),

    diab_true_early =
      !is.na(diab_date_pers) &
      (
        is.na(diab_date_classic) |
        diab_date_pers < diab_date_classic
      ))

In [ ]:
ct_prediab_true_early <- sum(early_warning_summ$prediab_true_early)
ct_diab_true_early <- sum(early_warning_summ$diab_true_early)

cat("Percentage prediab true early warning:", (ct_prediab_true_early / n_distinct(early_warning_summ$person_id)) * 100, "%\n")
cat("Percentage diab true early warning:", (ct_diab_true_early / n_distinct(early_warning_summ$person_id)) * 100, "%\n")

#### **Updated Individual-Level Summary - With Exclusions**

Exclude individuals where first measurement > 6.5 for all analyses.
Exclude individuals where first measurement > 5.7 for prediabetes analyses.

In [ ]:
# For now, just excluding individuals from the non-normal subset
#TODO: Later, revisit and make exlusion earlier if want to keep
#TODO: Later revisit and see if catching technical outliers and not truly
# high first measurements?
id_list_filt <- hba1c_anal_thresh %>%
  arrange(person_id, datetime) %>%
  group_by(person_id) %>%
  filter(first(hba1c) <= diab) %>%
  ungroup() %>%
  dplyr::pull(person_id)

n_distinct(id_list_filt)

In [ ]:
id_list_filt_prediab <- hba1c_anal_thresh %>%
  arrange(person_id, datetime) %>%
  group_by(person_id) %>%
  filter(first(hba1c) <= prediab) %>%
  ungroup() %>%
  dplyr::pull(person_id)

n_distinct(id_list_filt_prediab)

In [ ]:
early_warning_summ_diab_filt <- early_warning_summ %>%
    dplyr::filter(person_id %in% id_list_filt)
dim(early_warning_summ_diab_filt)

early_warning_summ_prediab_filt <- early_warning_summ %>%
    dplyr::filter(person_id %in% id_list_filt_prediab)
dim(early_warning_summ_prediab_filt)

In [ ]:
ct_prediab_filt_true_early <- sum(early_warning_summ_prediab_filt$prediab_true_early)
print(ct_prediab_filt_true_early)
ct_diab_filt_true_early <- sum(early_warning_summ_diab_filt$diab_true_early)
print(ct_diab_filt_true_early)

cat("Percentage prediab true early warning:", (ct_prediab_filt_true_early / n_distinct(early_warning_summ_prediab_filt$person_id)) * 100, "%\n")
cat("Percentage diab true early warning:", (ct_diab_filt_true_early / n_distinct(early_warning_summ_diab_filt$person_id)) * 100, "%\n")

#### **Example Individual Plot like CCPM**

TODO: do for now without exclusions?

In [ ]:
# Find good example
    # less than 10 measurements
    # true early warning
    # shift > 0.2?
early_warning_summ_list <- early_warning_summ %>%
    dplyr::filter(prediab_true_early == TRUE & diab_true_early == TRUE) %>%
    dplyr::pull(person_id)

plot_df <- hba1c_anal_thresh %>%
    dplyr::filter(person_id %in% early_warning_summ_list) %>%
    dplyr::filter(shift > 0.2) %>%
    dplyr::group_by(person_id) %>%
    dplyr::filter(n() < 10) %>%
    dplyr::arrange(datetime, .by_group = TRUE) %>%
    dplyr::mutate(measure_order = row_number()) %>%
    dplyr::ungroup()

In [ ]:
print(plot_df)

In [ ]:
id <- "2072208"
cohort_ids <- unlist(knn_cohort_t[, id], use.names = F)

group_data <- hba1c_anal_thresh %>%
    dplyr::filter(person_id %in% cohort_ids)

pers_prediab <- unique(hba1c_anal_thresh$pers_prediab[hba1c_anal_thresh$person_id == id])
pers_diab <- unique(hba1c_anal_thresh$pers_diab[hba1c_anal_thresh$person_id == id])
shift_val <- unique(hba1c_anal_thresh$shift[hba1c_anal_thresh$person_id == id])

# full_d <- density(hba1c_anal$hba1c, na.rm = TRUE)
# full_d_peak <- full_d$x[which.max(full_d$y)]
cohort_d <- density(group_data$hba1c, na.rm = TRUE)
cohort_d_peak <- cohort_d$x[which.max(cohort_d$y)]

In [ ]:
# Colors based on your description/image
# Grey for Global, Blue for Cohort
fill_colors <- c("All" = "#D3D3D3", "Cohort" = "#8FBBD9")
# Darker outlines
line_colors <- c("All" = "#888888", "Cohort" = "#4A7EBB")

# Threshold colors
col_pre <- "#B8860B" # Dark Gold
col_diab <- "#8B0000" # Dark Red

# Prep threshold lines
vlines <- data.frame(
  x = c(prediab, diab, pers_prediab, pers_diab),
  threshold = c("Classic Pre", "Classic Diab", "Cohort Pre", "Cohort Diab")
)

options(repr.plot.width = 12, repr.plot.height = 6)

ggplot() +
    geom_density(
        data = hba1c_anal_thresh,
        aes(x = hba1c, fill = "All"),
        color = "#888888",
        alpha = 0.4,
        size = 0.8) +
    geom_density(
        data = group_data,
        aes(x = hba1c, fill = "Cohort"),
        color = "#4A7EBB",
        alpha = 0.6,
        size = 0.8) +
    geom_point(
      data = hba1c_anal_thresh %>% dplyr::filter(person_id == id),
      aes(x = hba1c, y = 0),
      color = "black",
      size = 4,) +
    geom_text(
      data = plot_df %>% dplyr::filter(person_id == id),
      aes(x = hba1c, y = 0, label = measure_order),
      vjust = -0.5,
      size = 6,
      color = "black"
    ) +
    geom_vline(
        data = vlines,
        aes(xintercept = x, color = threshold, linetype = threshold),
        size = 1) +

  scale_fill_manual(
    name = "Distribution",
    values = fill_colors
  ) +
  scale_color_manual(
    name = "Thresholds",
    breaks = c("Classic Pre", "Classic Diab", "Cohort Pre", "Cohort Diab"),
    values = c(
      "Classic Pre" = col_pre,
      "Classic Diab" = col_diab,
      "Cohort Pre" = col_pre,
      "Cohort Diab" = col_diab
    ),
    labels = c(
      paste0("Classic Pre ", round(prediab, 2), "%"),
      paste0("Classic Diab ", round(diab, 2), "%"),
      paste0("Cohort Pre ", round(pers_prediab, 2), "%"),
      paste0("Cohort Diab ", round(pers_diab, 2), "%")
    )
  ) +
  scale_linetype_manual(
    name = "Thresholds",
    breaks = c("Classic Pre", "Classic Diab", "Cohort Pre", "Cohort Diab"),
    values = c(
      "Classic Pre" = "solid",
      "Classic Diab" = "solid",
      "Cohort Pre" = "dashed",
      "Cohort Diab" = "dashed"
    ),
    labels = c(
      paste0("Classic Pre ", round(prediab, 2), "%"),
      paste0("Classic Diab ", round(diab, 2), "%"),
      paste0("Cohort Pre ", round(pers_prediab, 2), "%"),
      paste0("Cohort Diab ", round(pers_diab, 2), "%")
    )
  ) +
  scale_x_continuous(breaks = seq(4, 9, 0.5), limits = c(3.5, 9)) +
  labs(
    title = "HbA1c Density: Global vs. Left-Shifted Cohort",
    subtitle = sprintf(
      "Shift: %.2f%% (Global Peak %.2f - Cohort Peak %.2f)",
      shift_val, full_d_peak, cohort_d_peak
    ),
    x = "HbA1c (%)",
    y = "Density"
  ) +
  theme_minimal()

In [ ]:
shift_val
full_d_peak
cohort_d_peak